# 02 — Fine-tuning ViT Pretrained dengan Hugging Face

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riki-profile/ai-engineer-journey/blob/main/05-vision-transformer/02_finetune_vit_hf.ipynb)

Notebook 01 menunjukkan ViT butuh data besar. Solusinya sama seperti di modul 02: **transfer learning**. Kali ini kita memakai ekosistem **Hugging Face** (`transformers`), yang juga akan dipakai di modul NLP dan LLM berikutnya.

Isi notebook:
1. Memuat ViT pretrained dan *image processor*-nya dari Hugging Face Hub.
2. Fine-tuning di **Flowers102**, dataset yang sama dengan modul 02, supaya bisa dibandingkan dengan ResNet18.
3. **Mixed precision** (AMP) agar training lebih cepat dan hemat memori di GPU.
4. Visualisasi **attention map** dari model pretrained.
5. Menyimpan model, dan (opsional) mengunggahnya ke Hugging Face Hub dengan token dari **Colab Secrets**.

> ⚖️ **Lisensi:** `transformers` berlisensi Apache-2.0; bobot `google/vit-base-patch16-224-in21k` juga Apache-2.0.

## 0. Cek GPU

Di Colab pilih **Runtime → Change runtime type → T4 GPU**, lalu jalankan ulang dari atas. Notebook tetap bisa berjalan di CPU, tetapi jauh lebih lambat.

In [ ]:
import shutil
import subprocess
import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"✅ GPU terdeteksi: {props.name} | memori {props.total_memory / 1e9:.1f} GB")
else:
    print("⚠️ GPU tidak terdeteksi, akan berjalan di CPU (lebih lambat).")
    print("   Di Colab: Runtime → Change runtime type → T4 GPU")

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"])

## 1. Instalasi dan Konfigurasi

`transformers` biasanya sudah ada di Colab; sel ini memastikan versinya cukup baru.

Model yang dipakai: **`google/vit-base-patch16-224-in21k`**, yaitu ViT-Base (86 juta parameter, patch 16, input 224×224) yang di-pretrain di ImageNet-21k (14 juta gambar, 21 ribu kelas). Model ini belum punya head klasifikasi untuk tugas tertentu, sehingga cocok untuk fine-tuning.

In [ ]:
%pip install -q "transformers>=4.44" 

Import dan konfigurasi. `QUICK_RUN=1` memakai subset sangat kecil dan 1 epoch.

In [ ]:
import os
import time
import copy
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from transformers import AutoImageProcessor, AutoModelForImageClassification, get_cosine_schedule_with_warmup

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

QUICK_RUN = os.environ.get("QUICK_RUN") == "1"
MODEL_NAME = "google/vit-base-patch16-224-in21k"
BATCH_SIZE = 32
EPOCHS = 1 if QUICK_RUN else 8
LR_BACKBONE, LR_HEAD = 3e-5, 1e-3
DATA_DIR = "./data"
USE_AMP = device == "cuda"        # mixed precision hanya bermanfaat di GPU
print(f"device: {device} | model: {MODEL_NAME} | epochs: {EPOCHS} | AMP: {USE_AMP}")

## 2. Image Processor dan Transformasi

Setiap model di Hub punya **image processor** yang menyimpan cara pra-pemrosesan saat model dilatih: ukuran input, `mean`, dan `std` normalisasi. Kita **baca nilainya dari processor** (bukan menulis angka manual) lalu membangun transformasi `torchvision` dengan augmentasi seperti di modul 02.

Perhatikan bahwa ViT ini memakai normalisasi `mean = std = 0.5`, berbeda dengan statistik ImageNet yang dipakai ResNet. Memakai normalisasi yang salah adalah bug yang sering tidak disadari.

In [ ]:
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
size = processor.size.get("height", processor.size.get("shortest_edge", 224))
MEAN, STD = processor.image_mean, processor.image_std
print(f"ukuran input: {size} | mean: {MEAN} | std: {STD}")

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(size, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize(int(size * 256 / 224)),
    transforms.CenterCrop(size),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

## 3. Dataset Flowers102

102 jenis bunga dengan hanya 10 gambar per kelas di split train, persis seperti modul 02.

In [ ]:
train_set = datasets.Flowers102(DATA_DIR, split="train", download=True, transform=train_tf)
val_set = datasets.Flowers102(DATA_DIR, split="val", download=True, transform=eval_tf)
test_set = datasets.Flowers102(DATA_DIR, split="test", download=True, transform=eval_tf)
CLASS_NAMES = train_set.classes
N_CLASSES = len(CLASS_NAMES)

if QUICK_RUN:
    g = torch.Generator().manual_seed(0)
    train_set = Subset(train_set, torch.randperm(len(train_set), generator=g)[:48].tolist())
    val_set = Subset(val_set, torch.randperm(len(val_set), generator=g)[:24].tolist())
    test_set = Subset(test_set, torch.randperm(len(test_set), generator=g)[:24].tolist())

loader_args = dict(batch_size=BATCH_SIZE, num_workers=2, pin_memory=(device == "cuda"))
train_loader = DataLoader(train_set, shuffle=True, **loader_args)
val_loader = DataLoader(val_set, **loader_args)
test_loader = DataLoader(test_set, **loader_args)
print(f"kelas: {N_CLASSES} | train: {len(train_set)} | val: {len(val_set)} | test: {len(test_set)}")

## 4. Memuat Model

`AutoModelForImageClassification.from_pretrained` memuat backbone ViT pretrained dan **membuat head klasifikasi baru** berukuran `N_CLASSES`. Peringatan bahwa sebagian bobot "baru diinisialisasi" (*newly initialized*) memang diharapkan: itu adalah head yang akan kita latih.

`id2label` dan `label2id` disimpan di konfigurasi model, sehingga saat model disimpan dan dipakai lagi, output-nya langsung berupa nama kelas.

In [ ]:
id2label = {i: n for i, n in enumerate(CLASS_NAMES)}
model = AutoModelForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=N_CLASSES,
    id2label=id2label,
    label2id={n: i for i, n in id2label.items()},
    ignore_mismatched_sizes=True,
).to(device)

n_total = sum(p.numel() for p in model.parameters())
n_head = sum(p.numel() for p in model.classifier.parameters())
print(f"Total parameter: {n_total / 1e6:.1f} juta | head baru: {n_head:,}")
print("Konfigurasi:", {k: getattr(model.config, k) for k in ["patch_size", "hidden_size", "num_hidden_layers", "num_attention_heads"]})

### 4.1 Format output model Hugging Face

Model HF menerima `pixel_values` dan mengembalikan objek berisi `logits`. Jika `labels` ikut diberikan, loss cross-entropy langsung dihitung di `outputs.loss`.

In [ ]:
x, y = next(iter(train_loader))
with torch.no_grad():
    out = model(pixel_values=x[:4].to(device), labels=y[:4].to(device))
print("pixel_values:", tuple(x[:4].shape))
print("logits      :", tuple(out.logits.shape))
print(f"loss        : {out.loss.item():.3f} (≈ ln(102) = 4.62 untuk head acak)")

## 5. Training dengan Mixed Precision

**Mixed precision (AMP)**: sebagian besar operasi dihitung dalam `float16` (lebih cepat dan hemat memori di GPU seperti T4), sedangkan bagian yang sensitif tetap `float32`.
- `torch.autocast` memilih presisi per operasi secara otomatis.
- `GradScaler` mengalikan loss dengan faktor besar sebelum `backward` agar gradien kecil tidak menjadi nol (*underflow*) di `float16`, lalu membaginya kembali sebelum `optimizer.step`.

Learning rate dibedakan untuk backbone dan head (seperti modul 02), ditambah warmup + cosine per langkah dari `transformers`.

In [ ]:
backbone_params = [p for n, p in model.named_parameters() if not n.startswith("classifier")]
optimizer = torch.optim.AdamW([
    {"params": backbone_params, "lr": LR_BACKBONE},
    {"params": model.classifier.parameters(), "lr": LR_HEAD},
], weight_decay=0.05)
total_steps = EPOCHS * len(train_loader)
scheduler = get_cosine_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
scaler = torch.amp.GradScaler(enabled=USE_AMP)


def run_epoch(loader, train):
    model.train(train)
    total_loss, benar, n = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                out = model(pixel_values=x, labels=y)
            if train:
                scaler.scale(out.loss).backward()
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                scheduler.step()
            total_loss += out.loss.item() * x.size(0)
            benar += (out.logits.argmax(1) == y).sum().item()
            n += x.size(0)
    return total_loss / n, benar / n


history = {"train_acc": [], "val_acc": []}
best_acc, best_state = -1.0, None
mulai = time.time()
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader, train=False)
    history["train_acc"].append(tr_acc); history["val_acc"].append(va_acc)
    if va_acc > best_acc:
        best_acc, best_state = va_acc, copy.deepcopy(model.state_dict())
    print(f"Epoch {epoch}/{EPOCHS} | train loss {tr_loss:.3f} acc {tr_acc:.3f} | val loss {va_loss:.3f} acc {va_acc:.3f}")

print(f"\nWaktu: {time.time() - mulai:.0f} detik | val acc terbaik: {best_acc:.3f}")
if torch.cuda.is_available():
    print(f"Memori GPU puncak: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
model.load_state_dict(best_state)

## 6. Evaluasi dan Perbandingan dengan Modul 02

Isi tabel perbandingan dengan hasil ResNet18 dari notebook `02-cnn/02_transfer_learning.ipynb` yang kamu jalankan sebelumnya. Pada dataset dengan sedikit data per kelas seperti ini, ViT yang di-pretrain di ImageNet-21k biasanya unggul, karena representasi hasil pretraining skala besar sangat berkualitas.

In [ ]:
test_loss, test_acc = run_epoch(test_loader, train=False)
print(f"ViT-B/16 (in21k) fine-tune → akurasi test: {test_acc:.3f}")
print("Bandingkan dengan ResNet18 di modul 02 (freeze / fine-tune semua).")

ep = range(1, EPOCHS + 1)
plt.plot(ep, history["train_acc"], "o-", label="train"); plt.plot(ep, history["val_acc"], "o-", label="val")
plt.xlabel("epoch"); plt.ylabel("akurasi"); plt.title("Fine-tuning ViT"); plt.legend(); plt.xticks(list(ep)); plt.show()

## 7. Attention Map dari Model Pretrained

Dengan `output_attentions=True`, model mengembalikan bobot attention setiap layer: tuple berisi 12 tensor berukuran `(batch, heads, 197, 197)`, yaitu 1 CLS + 196 patch.

Kita pakai **attention rollout** (Abnar & Zuidema, 2020): attention semua layer dikalikan berurutan (ditambah identitas untuk residual connection), sehingga terlihat aliran informasi dari patch input ke token CLS di output. Implementasi attention bawaan yang cepat (SDPA) tidak mengembalikan bobot attention, jadi untuk visualisasi kita beralih sementara ke implementasi `eager`.

In [ ]:
def attention_rollout(attentions):
    # attentions: tuple (layers) berisi (batch, heads, N, N)
    hasil = None
    for a in attentions:
        a = a.mean(dim=1)                                        # rata-rata head
        a = a + torch.eye(a.size(-1), device=a.device)           # residual connection
        a = a / a.sum(dim=-1, keepdim=True)
        hasil = a if hasil is None else a @ hasil
    return hasil[:, 0, 1:]                                       # CLS → setiap patch


model.set_attn_implementation("eager")
model.eval()
x, y = next(iter(test_loader))
x = x[:6]
with torch.no_grad():
    out = model(pixel_values=x.to(device), output_attentions=True)
rollout = attention_rollout(out.attentions).cpu()
n_side = int(rollout.shape[-1] ** 0.5)
preds = out.logits.argmax(1).cpu()

mean_t, std_t = torch.tensor(MEAN).view(3, 1, 1), torch.tensor(STD).view(3, 1, 1)
fig, axes = plt.subplots(2, len(x), figsize=(2.6 * len(x), 5.5))
for j in range(len(x)):
    img = (x[j] * std_t + mean_t).clamp(0, 1).permute(1, 2, 0)
    peta = rollout[j].reshape(n_side, n_side)
    axes[0, j].imshow(img)
    axes[0, j].set_title(f"asli: {CLASS_NAMES[y[j]]}\npred: {CLASS_NAMES[preds[j]]}", fontsize=7,
                         color="green" if preds[j] == y[j] else "red")
    axes[1, j].imshow(img); axes[1, j].imshow(peta, cmap="jet", alpha=0.5, extent=(0, size, size, 0))
for a in axes.flat:
    a.axis("off")
plt.tight_layout(); plt.show()
model.set_attn_implementation("sdpa")   # kembali ke implementasi cepat

## 8. Menyimpan Model (dan Opsional: Unggah ke Hub)

`save_pretrained` menyimpan bobot dan konfigurasi (termasuk `id2label`) dalam format standar Hugging Face, sehingga bisa dimuat ulang dengan `from_pretrained(folder)`.

Untuk mengunggah ke Hugging Face Hub:
1. Buat token (*write*) di https://huggingface.co/settings/tokens.
2. Simpan di **Colab Secrets** dengan nama `HF_TOKEN`. **Jangan pernah** menulis token di notebook.
3. Set `PUSH_TO_HUB = True` dan isi `HUB_REPO` dengan nama repo milikmu.

In [ ]:
SAVE_DIR = "vit-flowers102"
model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
print("Disimpan:", sorted(os.listdir(SAVE_DIR)))

# Uji muat ulang
reloaded = AutoModelForImageClassification.from_pretrained(SAVE_DIR).to(device).eval()
with torch.no_grad():
    sama = torch.equal(reloaded(pixel_values=x[:2].to(device)).logits.argmax(1).cpu(), preds[:2])
print("Prediksi model yang dimuat ulang sama?", sama)

PUSH_TO_HUB = False
HUB_REPO = "nama-user-kamu/vit-flowers102"
if PUSH_TO_HUB:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except ImportError:
        token = os.getenv("HF_TOKEN")
    model.push_to_hub(HUB_REPO, token=token)
    processor.push_to_hub(HUB_REPO, token=token)
    print("Diunggah ke https://huggingface.co/" + HUB_REPO)

## Ringkasan

- `AutoImageProcessor` menyimpan pra-pemrosesan resmi model (ukuran, mean/std); ambil nilainya dari sana, jangan menebak.
- `AutoModelForImageClassification.from_pretrained(..., num_labels, id2label, ignore_mismatched_sizes=True)` membuat head baru untuk kelas kita.
- **Mixed precision** (`autocast` + `GradScaler`) mempercepat training dan menghemat memori GPU.
- **Attention rollout** memvisualisasikan bagian gambar yang memengaruhi keputusan model.
- `save_pretrained` / `push_to_hub` untuk menyimpan dan berbagi model; token selalu dari Secrets.

## Latihan

1. **Linear probe vs fine-tune.** Bekukan seluruh backbone (hanya `model.classifier` yang dilatih) dan bandingkan akurasi serta waktu per epoch dengan fine-tune penuh. Bandingkan juga dengan hasil freeze ResNet18 di modul 02. Model mana yang fiturnya lebih baik "apa adanya"?
2. **Efek mixed precision.** Jalankan 2 epoch dengan `USE_AMP = True` dan `False`. Catat waktu per epoch, memori GPU puncak (`torch.cuda.max_memory_allocated()`), dan akurasi. Berapa percepatannya di GPU T4?
3. **Model lain dari Hub.** Ganti `MODEL_NAME` dengan model lain yang ukurannya mirip, misalnya `facebook/deit-small-patch16-224` (DeiT, lebih kecil) atau `microsoft/swin-tiny-patch4-window7-224` (Swin, ViT hierarkis). Periksa lisensinya di halaman model, lalu bandingkan akurasi, jumlah parameter, dan waktu training dalam satu tabel.